In [0]:
#%pip install faker

In [0]:
#%restart_python

In [0]:
base_path="/Volumes/ext_novartis/default/files"

In [0]:
from faker import Faker
import random
import pandas as pd

fake = Faker()

num_doctors = 200000

specialties = [
"Cardiology","Neurology","Orthopedic",
"Endocrinology","Dermatology"
]

cities = [
"Chennai","Mumbai","Delhi",
"Hyderabad","Bangalore"
]

data = []

for i in range(num_doctors):

    data.append((
        f"D{i}",
        fake.name(),
        random.choice(specialties),
        random.choice(cities),
        random.randint(1,3)
    ))

hcp_df = spark.createDataFrame(
data,
["doctor_id","doctor_name","specialty","city","tier"]
)

hcp_df.write.mode("overwrite").parquet(f"{base_path}/hcp/")

In [0]:
hcp_df.show()

In [0]:
hospital_names = [
"Apollo Hospital",
"Apollo Hosp",
"Apollo Hospitals",
"Fortis Hospital",
"Fortis Hosp",
"AIIMS",
"CMC Hospital"
]

num_hospitals = 50000

data = []

for i in range(num_hospitals):

    data.append((
        f"H{i}",
        random.choice(hospital_names),
        random.choice(cities),
        random.choice(["Private","Government"])
    ))

hco_df = spark.createDataFrame(
data,
["hospital_id","hospital_name","city","hospital_type"]
)

hco_df.write.mode("overwrite").parquet(f"{base_path}/hco/")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
import random

relationships = []

for i in range(num_doctors):

    doctor = f"D{i}"
    hospital1 = f"H{random.randint(1,50000)}"

    relationships.append((doctor,hospital1,"2021-01-01",None))

    if random.random() < 0.2:
        hospital2 = f"H{random.randint(1,50000)}"
        relationships.append((doctor,hospital2,"2023-01-01",None))


schema = StructType([
    StructField("doctor_id", StringType(), True),
    StructField("hospital_id", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True)
])

rel_df = spark.createDataFrame(relationships, schema)

rel_df.write.mode("overwrite").parquet(f"{base_path}/hcp_hco_relationship/")

In [0]:
from pyspark.sql.functions import expr

calls_df = spark.range(0,100000000)

calls_df = calls_df.withColumn(
    "doctor_id",
    expr("concat('D', cast(rand()*200000 as int))")
)

calls_df = calls_df.withColumn(
    "rep_id",
    expr("concat('R', cast(rand()*1000 as int))")
)

calls_df = calls_df.withColumn(
    "call_channel",
    expr("""
    CASE
        WHEN rand() < 0.5 THEN 'InPerson'
        ELSE 'Virtual'
    END
    """)
)

calls_df.write.mode("overwrite").parquet(f"{base_path}/calls/")

In [0]:
emails_df = spark.range(0,300000000)

emails_df = emails_df.withColumn(
"doctor_id",
expr("concat('D', cast(rand()*200000 as int))")
)

emails_df = emails_df.withColumn(
"engagement_type",
expr("""
CASE
WHEN rand() < 0.4 THEN 'email_open'
WHEN rand() < 0.7 THEN 'email_click'
ELSE 'unsubscribe'
END
""")
)

emails_df.write.mode("overwrite").parquet(f"{base_path}/emails/")

In [0]:
emails_df = spark.range(0,300000000)

emails_df = emails_df.withColumn(
"doctor_id",
expr("concat('D', cast(rand()*200000 as int))")
)

emails_df = emails_df.withColumn(
"engagement_type",
expr("""
CASE
WHEN rand() < 0.4 THEN 'email_open'
WHEN rand() < 0.7 THEN 'email_click'
ELSE 'unsubscribe'
END
""")
)

emails_df.write.mode("overwrite").parquet(f"{base_path}/emails/")

In [0]:
impressions_df = spark.range(0,500000000)

impressions_df = impressions_df.withColumn(
"doctor_id",
expr("concat('D', cast(rand()*200000 as int))")
)

impressions_df = impressions_df.withColumn(
"device",
expr("""
CASE
WHEN rand() < 0.4 THEN 'Mobile'
WHEN rand() < 0.7 THEN 'Desktop'
ELSE 'Tablet'
END
""")
)

impressions_df.write.mode("overwrite").parquet(f"{base_path}/ad_impressions/")

In [0]:
late_events = emails_df.sample(0.02)

late_events = late_events.withColumn(
"event_delay_days",
expr("cast(rand()*5 as int)")
)
late_events.write.mode("overwrite").parquet(f"{base_path}/late_events/")

In [0]:
cdc_calls = calls_df.sample(0.01)

cdc_calls = cdc_calls.withColumn(
"cdc_operation",
expr("""
CASE
WHEN rand() < 0.7 THEN 'UPDATE'
ELSE 'DELETE'
END
""")
)

cdc_calls.write.mode("overwrite").parquet("/Volumes/novartis/bp2child/files/cdc_calls/")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
bad_records = spark.range(10000)

bad_records = bad_records.withColumn(
"doctor_id",
lit(None)
)

bad_records.write.mode("overwrite").parquet(f"{base_path}/bad_data/")